In [9]:
from sqlalchemy import create_engine
import pandas as pd
import yaml
import os

filename = os.path.join(os.getcwd(), 'creds.yaml')


with open(filename, "r") \
      as file:
    creds = yaml.safe_load(file)


def reads_from_mysql(creds, query):
    """
    Returns as dataframe the result of a query to a MySQL database.

    Args:
        creds (dict): The credentials to access the database.
        query (string): The query.

    Returns:
        pandas dataframe: the output table of the query.
    """
    _db_user = creds['username']
    _db_password = creds['password']
    _db_host = creds['host']
    _db_name = creds['database']
    engine = create_engine(f"mysql://{_db_user}:{_db_password}@{_db_host}:3306/{_db_name}")
    df = pd.read_sql(query, engine)
    return df

def write_to_database(creds, df, table_name, if_exists='append'):
    """
    Returns as dataframe the result of a query to a MySQL database.

    Args:
        creds (dict): The credentials to access the database.
        query (string): The query.

    Returns:
        pandas dataframe: the output table of the query.
    """
    _db_user = creds['username']
    _db_password = creds['password']
    _db_host = creds['host']
    _db_name = creds['database']
    engine = create_engine(f"mysql://{_db_user}:{_db_password}@{_db_host}:3306/{_db_name}")
    with engine.connect() as connection:
        df.to_sql(table_name, con=connection, if_exists=if_exists, index=False) 


In [2]:
import pandas as pd

In [62]:
df = pd.read_csv('C:/Users/massi/OneDrive/Desktop/Uni/DMFBI-R/4.nocode/data/invoices_eae.csv',sep=';')

In [63]:
write_to_database(creds=creds['mysql-db'], df=df, table_name='python_test')

ASSIGNMENT

In [10]:
meteo_types = {'temperature':'float64','relative_humidity':'float64','precipitation_rate':'float64','wind_speed':'float64','zipcode':'str'}
contracts_types = {'CONTRACT_ID':'int64','CLIENT_TYPE_ID':'int64','AVG_EUROS_IMPORT':'float64','POWER_P1':'float64','HAS_GAS':'boolean','HAS_SOLAR':'boolean','ZIPCODE':'str'}
zipcode_types = {'ZIPCODE':'str','ZC_LATITUDE':'float64','ZC_LONGITUDE':'float64','AUTONOMOUS_COMMUNITY':'str','AUTONOMOUS_COMMUNITY_NK':'str','PROVINCE':'str'}

In [11]:
def _filter_data_isin(table: str, column: str, lookup: list):
    '''
    Arg:
        table -> table you want to filter
        column -> table column as input for the lookup
        lookup -> list or any iterable that contains lookup values
    '''
    return table[table[column].isin(lookup)]
df_contracts = pd.read_csv('contracts_eae.csv', dtype=contracts_types)
df_zipcode = pd.read_csv('zipcode_eae_v2.csv', dtype=zipcode_types)

In [12]:
df_contracts.columns = df_contracts.columns.str.lower()
df_zipcode.columns = df_zipcode.columns.str.lower()

In [13]:
df_zipcode.dtypes

zipcode                     object
zc_latitude                float64
zc_longitude               float64
autonomous_community        object
autonomous_community_nk     object
province                    object
dtype: object

In [14]:
zipcode_grouped = df_contracts.groupby('zipcode')['contract_id'].count()
zipcode_top = list(zipcode_grouped[zipcode_grouped > 10].index)

In [18]:
chunks = pd.read_csv('meteo_eae.csv', chunksize = 100000, delimiter=';', \
                        dtype = meteo_types, parse_dates= ['date'])


In [19]:
chunks

In [20]:

df_meteo_top_raw = pd.concat([_filter_data_isin(table=chunk, \
                            column='zipcode',lookup=zipcode_top) \
                            for chunk in chunks], ignore_index=True)


In [23]:
def _category_p(power: float) -> str:
    if power >= 5000:
        return 'Over 5 MW'
    elif power < 3000:
        return 'Under 3 MW'
    else:
        return 'Between 3 and 5 MW'
df_contracts['p1_category'] = df_contracts['power_p1'].apply(lambda x: _category_p(x))


In [24]:
df_contracts

,contract_id,client_type_id,avg_euros_import,power_p1,has_gas,has_solar,zipcode,p1_category
0,6951,0,132.22,5250.0,False,True,43203,Over 5 MW
1,7507,0,45.43,6400.0,False,True,08904,Over 5 MW
2,1815,0,67.30,4650.0,False,True,48800,Between 3 and 5 MW
3,5273,0,110.93,3950.0,False,False,08205,Between 3 and 5 MW
4,6079,0,73.46,4100.0,False,True,08032,Between 3 and 5 MW
...,...,...,...,...,...,...,...,...
8913,1150,0,29.96,3950.0,False,False,31007,Between 3 and 5 MW
8914,2897,0,48.17,4950.0,False,False,28002,Between 3 and 5 MW
8915,405,0,103.79,2950.0,False,False,24004,Under 3 MW
8916,1958,0,47.58,4100.0,False,False,03570,Between 3 and 5 MW


In [25]:
df_contracts_zero_raw = df_contracts[df_contracts['client_type_id']==0]

In [26]:
df_contracts_zero = df_contracts_zero_raw[['p1_category','zipcode','has_solar']]
df_meteo_top = df_meteo_top_raw[['date','temperature','relative_humidity','zipcode']]

In [27]:
df_solar_indicators_raw = df_contracts_zero.merge(df_meteo_top, how='right', left_on='zipcode', \
                                                    right_on='zipcode', )

In [32]:
df_solar_indicators_raw

,p1_category,zipcode,has_solar,temperature,relative_humidity,year,month
0,Between 3 and 5 MW,03540,False,14.27874,59.61995,2024,January
1,Between 3 and 5 MW,03540,False,14.27874,59.61995,2024,January
2,Over 5 MW,03540,False,14.27874,59.61995,2024,January
3,Between 3 and 5 MW,03540,False,14.27874,59.61995,2024,January
4,Over 5 MW,03540,False,14.27874,59.61995,2024,January
...,...,...,...,...,...,...,...
1293439,Between 3 and 5 MW,50015,True,5.99634,59.47399,2024,December
1293440,Between 3 and 5 MW,50015,False,5.99634,59.47399,2024,December
1293441,Over 5 MW,50015,False,5.99634,59.47399,2024,December
1293442,Over 5 MW,50015,True,5.99634,59.47399,2024,December


In [29]:
df_solar_indicators_raw['year'] = df_solar_indicators_raw['date'].dt.strftime("%Y")
df_solar_indicators_raw['month'] = df_solar_indicators_raw['date'].dt.strftime("%B")

In [31]:
df_solar_indicators_raw = df_solar_indicators_raw.drop(columns='date')

In [33]:
df_solar_indicators_raw = df_solar_indicators_raw.groupby(['year','month','zipcode','p1_category','has_solar']).agg( \
                                {'temperature':['min', 'max'],'relative_humidity':'mean'} \
                                ).reset_index()

In [36]:
df_solar_indicators_raw

,year,month,zipcode,p1_category,has_solar,temperature_min,temperature_max,relative_humidity_mean
0,2024,April,03540,Between 3 and 5 MW,False,13.47221,19.76023,59.467606
1,2024,April,03540,Between 3 and 5 MW,True,13.47221,19.76023,59.467606
2,2024,April,03540,Over 5 MW,False,13.47221,19.76023,59.467606
3,2024,April,03540,Over 5 MW,True,13.47221,19.76023,59.467606
4,2024,April,03540,Under 3 MW,False,13.47221,19.76023,59.467606
...,...,...,...,...,...,...,...,...
9355,2024,September,48004,Under 3 MW,False,12.92200,20.30558,76.806422
9356,2024,September,50015,Between 3 and 5 MW,False,14.64518,24.72549,58.935070
9357,2024,September,50015,Between 3 and 5 MW,True,14.64518,24.72549,58.935070
9358,2024,September,50015,Over 5 MW,False,14.64518,24.72549,58.935070


In [121]:
df_solar_indicators_raw.columns

Index(['year', 'month', 'zipcode', 'p1_category', 'has_solar',
       'temperature_min', 'temperature_max', 'relative_humidity_mean'],
      dtype='object')

In [35]:
df_solar_indicators_raw.columns = ['_'.join(col).strip('_') for col in df_solar_indicators_raw.columns]

In [38]:
FINAL_COLS = ['year', 'month', 'zipcode', 'p1_category',  \
           'temperature_min', 'temperature_max', 'relative_humidity_mean']


In [39]:
solar_indicators_with_solar = df_solar_indicators_raw[df_solar_indicators_raw['has_solar']==True][FINAL_COLS]

In [40]:
solar_indicators_no_solar = df_solar_indicators_raw[df_solar_indicators_raw['has_solar']==False][FINAL_COLS]

In [41]:
solar_indicators_with_solar.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4176 entries, 1 to 9359
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    4176 non-null   object 
 1   month                   4176 non-null   object 
 2   zipcode                 4176 non-null   object 
 3   p1_category             4176 non-null   object 
 4   temperature_min         4176 non-null   float64
 5   temperature_max         4176 non-null   float64
 6   relative_humidity_mean  4176 non-null   float64
dtypes: float64(3), object(4)
memory usage: 261.0+ KB


In [42]:
solar_indicators_no_solar.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5184 entries, 0 to 9358
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    5184 non-null   object 
 1   month                   5184 non-null   object 
 2   zipcode                 5184 non-null   object 
 3   p1_category             5184 non-null   object 
 4   temperature_min         5184 non-null   float64
 5   temperature_max         5184 non-null   float64
 6   relative_humidity_mean  5184 non-null   float64
dtypes: float64(3), object(4)
memory usage: 324.0+ KB
